In [1]:
import sys
print(sys.executable)

f:\Study ITU\Research Project\ecpfs\.venv\Scripts\python.exe


In [2]:
from pathlib import Path
from ecpfs import Builder, Index, Metric

print("eCP-FS import successful!")
print("Working directory:", Path.cwd())

eCP-FS import successful!
Working directory: f:\Study ITU\Research Project\ecpfs\mycode


In [3]:
from pathlib import Path
import h5py

# Locate the dataset
data_path = Path.cwd().parent / "data" / "ccnews-nomic-768-normalized.h5"

print("Dataset exists:", data_path.exists())

# Open and inspect the dataset
with h5py.File(data_path, "r") as f:
    print("Train shape:", f["train"].shape)
    print("Test shape:", f["test"].shape)
    print("First 5 values:", f["train"][0, :5])

Dataset exists: True
Train shape: (495328, 768)
Test shape: (1000, 768)
First 5 values: [ 0.00118432  0.04939296 -0.21632339  0.05403711  0.07347868]


In [4]:
import h5py

with h5py.File(data_path, "r") as f:

    print("=== DATASET INFORMATION ===")

    # Display dataset metadata
    for key, value in f.attrs.items():
        print(f"{key}: {value}")

    print("\n=== DATASET NAMES AND STRUCTURE ===")

    # Display every dataset name, shape and data type
    for name in f.keys():
        dataset = f[name]

        print(f"\nName: {name}")
        print(f"Shape: {dataset.shape}")
        print(f"Data type: {dataset.dtype}")

    print("\n=== EXAMPLE VALUES ===")

    print("First train vector:", f["train"][0, :10])
    print("First test vector:", f["test"][0, :10])
    print("First 10 neighbor IDs:", f["neighbors"][0, :10])
    print("First 10 distances:", f["distances"][0, :10])
    print("First average distance:", f["avg_distances"][0])

=== DATASET INFORMATION ===
dimension: 768
distance: normalized
point_type: float

=== DATASET NAMES AND STRUCTURE ===

Name: avg_distances
Shape: (1000,)
Data type: float64

Name: distances
Shape: (1000, 100)
Data type: float64

Name: neighbors
Shape: (1000, 100)
Data type: int64

Name: test
Shape: (1000, 768)
Data type: float32

Name: train
Shape: (495328, 768)
Data type: float32

=== EXAMPLE VALUES ===
First train vector: [ 0.00118432  0.04939296 -0.21632339  0.05403711  0.07347868  0.04265145
  0.0213018   0.05593102  0.03132028 -0.00196485]
First test vector: [-0.01059632  0.00122034 -0.20069289 -0.05815577 -0.01281362 -0.04921297
 -0.04323493 -0.04478858  0.00307377 -0.01775028]
First 10 neighbor IDs: [331646 460655 118234  80550 218017 269547 362345 346445  32329 306328]
First 10 distances: [0.14955997 0.1539191  0.15749824 0.16052967 0.17653155 0.18299526
 0.18959135 0.19473642 0.1959163  0.19612479]
First average distance: 0.4515838623046875


Name	Meaning	Purpose in research
train	495,328 vectors, each with 768 dimensions	Used to build the eCP-FS index
test	1,000 query vectors, each with 768 dimensions	Used to search the index
neighbors	IDs of the 100 true nearest neighbours for each query	Used to calculate recall
distances	Actual distances of those 100 nearest neighbours	Used to evaluate distance results
avg_distances	Average distance from each query to all corpus vectors	Useful for understanding dataset/query characteristics

In [5]:
import numpy as np

with h5py.File(data_path, "r") as f:
    train_vector = f["train"][0]
    test_vector = f["test"][0]

print("Train vector length:", np.linalg.norm(train_vector))
print("Test vector length:", np.linalg.norm(test_vector))

Train vector length: 1.0
Test vector length: 1.0000001


In [6]:
with h5py.File(data_path, "r") as f:
    query = f["test"][0]
    neighbor_id = int(f["neighbors"][0, 0])
    neighbor = f["train"][neighbor_id]
    recorded_distance = f["distances"][0, 0]

# Calculate different distances
euclidean = np.linalg.norm(query - neighbor)
squared_euclidean = euclidean ** 2

cosine_distance = 1 - (
    np.dot(query, neighbor) /
    (np.linalg.norm(query) * np.linalg.norm(neighbor))
)

print("Neighbor ID:", neighbor_id)
print("Recorded distance:", recorded_distance)
print("Euclidean distance:", euclidean)
print("Squared Euclidean:", squared_euclidean)
print("Cosine distance:", cosine_distance)

Neighbor ID: 331646
Recorded distance: 0.14955997467041016
Euclidean distance: 0.54691863
Squared Euclidean: 0.29911998
Cosine distance: 0.14955997


In [7]:
import numpy as np
import h5py

with h5py.File(data_path, "r") as f:
    for name in ["train", "test"]:

        dataset = f[name]
        total_checked = 0
        not_normalized = 0
        max_difference = 0

        # Read 10,000 vectors at a time
        for start in range(0, len(dataset), 10000):
            vectors = dataset[start:start + 10000]

            lengths = np.linalg.norm(vectors, axis=1)

            differences = np.abs(lengths - 1.0)

            not_normalized += np.count_nonzero(
                ~np.isfinite(lengths) | (differences > 0.0001)
            )

            max_difference = max(
                max_difference,
                float(np.max(differences))
            )

            total_checked += len(vectors)

        print(f"\nDataset: {name}")
        print("Total vectors checked:", total_checked)
        print("Vectors not normalized:", not_normalized)
        print("Maximum difference from 1:", max_difference)


Dataset: train
Total vectors checked: 495328
Vectors not normalized: 0
Maximum difference from 1: 1.1920928955078125e-07

Dataset: test
Total vectors checked: 1000
Vectors not normalized: 0
Maximum difference from 1: 1.1920928955078125e-07


In [8]:
print(Builder.__doc__)

Builds a new eCP index in three steps: create the ``Builder``, call
``select_representatives`` or ``select_representatives_custom``, then call
``build``. Every method releases the GIL while it works.

The index is created at ``index_path``. ``levels`` is the number of node
levels below the root, and ``is_normalized`` should be set only if every
embedding is unit-length. ``memory_limit_bytes`` is a target for the build's
memory use rather than a hard cap, and defaults to 80% of system RAM. Leave
``embedding_dtype`` as None to store each file's own dtype; a narrower one
logs a warning. ``rep_chunk_bytes`` is the chunk size for the representative
arrays and ``node_chunk_bytes`` the chunk size for the tree nodes. Measure
zarr read speed at a few chunk sizes on your own data before changing them.
